In [0]:
# ======================================
# Dataset: olist_products
# Layer: Silver
# Source: Bronze (Delta)
# Obj: 1 row per product_id
# Depende de: product_category_name_translation (conceptual)
# product_category_name puede ser null según dataset original (Olist).
# ======================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
%run ../commons/silver_utils

In [0]:
%run ../../00_config/paths

In [0]:
df_bronze = spark.read.format("delta").load(BRONZE_PRODUCTS_PATH)
df_bronze.printSchema()

In [0]:
df = normalize_columns(df_bronze)

In [0]:
df = df.select(
    F.col("product_id"),
    F.trim(F.lower(F.col("product_category_name"))).alias("product_category_name"),
    F.col("product_name_lenght").cast(IntegerType()).alias("product_name_lenght"),
    F.col("product_description_lenght").cast(IntegerType()).alias("product_description_lenght"),
    F.col("product_photos_qty").cast(IntegerType()).alias("product_photos_qty"),
    F.col("product_weight_g").cast(IntegerType()).alias("product_weight_g"),
    F.col("product_length_cm").cast(IntegerType()).alias("product_length_cm"),
    F.col("product_height_cm").cast(IntegerType()).alias("product_height_cm"),
    F.col("product_width_cm").cast(IntegerType()).alias("product_width_cm")
)

In [0]:
df = df.filter(F.col("product_id").isNotNull())


In [0]:
df = df.filter(
    (F.col("product_weight_g").isNull() | (F.col("product_weight_g") > 0)) &
    (F.col("product_length_cm").isNull() | (F.col("product_length_cm") > 0)) &
    (F.col("product_height_cm").isNull() | (F.col("product_height_cm") > 0)) &
    (F.col("product_width_cm").isNull() | (F.col("product_width_cm") > 0))
)


In [0]:
df = df.filter(
    (F.col("product_photos_qty").isNull()) |
    (F.col("product_photos_qty") >= 0)
)


In [0]:
df = df.dropDuplicates(["product_id"])

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df.count())


In [0]:
df.printSchema()

In [0]:
write_silver(df, SILVER_PRODUCTS_PATH)